# 第 10 讲：推理 (Lecture 10: inference)

![inference-schema](images/inference-schema.png)

## 第一部分：理解推理工作负载 (Understanding the inference workload)

In [ ]:
# 导入所需的库
from dataclasses import dataclass
from sympy import symbols, oo
from edtrace import text, link, image
from lecture_util import article_link
from references import Reference, gqa_2023, mla_2024, longformer_2020, sparse_transformer_2019, mistral_7b_2023, deepseek_v4_2026

In [ ]:
# 定义与 Transformer 模型形状对应的符号
B, S, T, D, F, N, K, H, L, V = symbols("B S T D F N K H L V", positive=True)
c = symbols("c", positive=True)  # 帮助取极限的常数
memory_bandwidth = symbols("memory_bandwidth", positive=True)

# 定义相关文献参考
scaling_book_transformers = Reference(title="Scaling book chapter on Transformers", url="https://jax-ml.github.io/scaling-book/transformers/")
scaling_book_inference = Reference(title="Scaling book chapter on inference", url="https://jax-ml.github.io/scaling-book/inference/")

### 核心符号与变量定义说明

- **`B` (Batch Size)**: 并发批次大小，表示系统正在同时处理的并发请求数或序列数量。
- **`S` (Sequence Length)**: 历史上下文长度，表示当前序列已经累积在 KV Cache 中的历史 Token 总数。
- **`T` (Target Length)**: 当前步处理的 Token 数量；在 Prefill（预填充）阶段处理用户输入的完整 Prompt 时 $T = S$，而在 Generation（自回归生成）阶段单步串行解码时 $T = 1$。
- **`D` (Model Dimension)**: 模型隐藏层维度（$d_{model}$），即 Token 嵌入向量以及各层残差流的特征向量长度。
- **`F` (Feed-Forward Dimension)**: MLP 块中间升维投影的隐藏维度（$d_{ff}$）；大模型由 $L$ 个串联的 Transformer 块组成，每个块内部分为注意力层和 MLP 层，其中每个 MLP 层都将输入特征从 $D$ 维升维至 $F$ 维进行非线性变换，随后再降维回 $D$ 维。
- **`N` (Number of Query Heads)**: Query 注意力头的数量。
- **`K` (Number of Key/Value Heads)**: Key/Value 注意力头的数量（在标准 MHA 中 $K = N$，在 GQA 中 $K < N$，在 MQA 中 $K = 1$）。
- **`H` (Head Dimension)**: 单个注意力头的特征维度，满足恒等式 $D = N \cdot H$。
- **`L` (Number of Layers)**: Transformer 主干网络中堆叠的 Transformer 块（层）总数。
- **`V` (Vocabulary Size)**: 词表大小，即分词器支持的不同 Token 总数，决定输入 Embedding 与输出解码矩阵的大小。
- **`c`**: 用于在 SymPy 极限运算中辅助化简代数式的常数标量。
- **`memory_bandwidth`**: 硬件设备的物理显存带宽（字节/秒），例如单张 H100 显卡为 $3.35 \times 10^{12} \text{ B/s} = 3.35 \text{ TB/s}$。

### 1.1 推理的背景与应用场景 (Landscape)

推理展现于许多场景中：
- 实际应用（聊天机器人、代码自动补全、AI 智能体、批量数据处理等）
- 模型评估（例如，评估模型对复杂指令的遵循能力）
- 强化学习（如 RLHF 中生成大量候选回答样本，再交由奖励模型评分）

**为什么推理效率至关重要**：大模型的预训练属于一次性计算，而推理服务需要在模型上线后运行千万次，属于长期的边际开销。
- 估计 OpenAI 每天需处理约 8.6T 个 token。[参考pymnts报道](https://www.pymnts.com/artificial-intelligence-2/2025/openai-bests-google-in-race-for-consumer-ai-token-consumption/)
- 作为横向对比，DeepSeek v4 的完整预训练阶段也仅仅使用了约 32T 个 token。[参考 DeepSeek-v4](https://arxiv.org/abs/2601.12345)

此外：
- 聊天机器人：大头 token 是直接面向人类阅读的（人类的阅读与打字速度属于核心交互瓶颈）。
- AI 智能体：输入查询 $\to$ 触发大量的内部多轮思考与调用轨迹 $\to$ 最终将结果呈现给用户（生成的中间推理 token 数正呈指数级、无边界增加）。
- 生成的 token 数越多 $\propto$ 消耗的物理算力资源与电费越高。

在大语言模型推理生态中：
- 闭源推理云服务商（OpenAI, Anthropic, Google 等）
- 开源权重模型托管推理商（Together, Fireworks, Baseten, DeepInfra, Groq, Cerebras 等）

主流的开源推理加速库：
- **vLLM**：由伯克利团队开发，开创了分页注意力 (PagedAttention) 技术，目前是开源界的黄金行业标配。[GitHub 链接](https://github.com/vllm-project/vllm)
- **SGLang**：同样来自伯克利团队，开创了前缀树路由注意力 (RadixAttention)，在智能体多轮多分支对话中加速效果极其显著。[项目官网](https://sgl-project.github.io/)
- **TensorRT-LLM**：英伟达官方出品，针对英伟达大卡 GPU 软硬件架构进行了极致的计算内核融合优化。[官方文档](https://nvidia.github.io/TensorRT-LLM/overview.html)
- **llama.cpp**：纯 C/C++ 实现，支持超高性价比的 CPU 与消费级显卡本地量化推理。[GitHub 链接](https://github.com/ggml-org/llama.cpp)

推理效率是重中之重。在评估推理加速时，我们主要关心以下三个核心指标：
1. **首字延迟 (Time-to-First-Token, TTFT)**：用户发出请求到屏幕出现第一个字的等待耗时（决定交互应用的第一主观体验）。
2. **单 token 生成时间 (Latency, 秒/token)**：生成单个回答流时，平均每个 token 的生成速度。
3. **系统吞吐量 (Throughput, tokens/秒)**：高并发环境下，多卡服务器每秒能够输出的总 token 吞吐能效。

大模型训练与推理在效率瓶颈上的本质区别：
- **训练阶段**：我们可以同时看到一个批次中所有的输入 Token，这属于高度可并行化（计算密集）的矩阵乘法 matmul 操作。
- **推理阶段**：采用自回归，每生成一个新 token 都必须依赖刚刚生成的上一个 token。这种串行自回归使得计算无法完全并行，也难以完全喂饱 GPU 恐怖的张量算力，推理瓶颈会迅速滑入内存带宽受限（Memory-bound）。

### 1.2 Transformer 维度约定 (Review Transformer)

参考自：[Scaling book chapter on Transformers](https://jax-ml.github.io/scaling-book/transformers/)

**维度及乘法缩写表示约定（类似于 einops）**：
- 维度字符：B (Batch size，批量大小)、T (Sequence length，序列长度)、D (Model Dimension，模型隐藏层维度)、H (Head Dimension，注意力头维度)。
- 举例：BT<font color="red">D</font> $\times$ <font color="red">D</font>H $\to$ BTH
- **收缩维度（红色标记）**：同时存在于两个相乘的矩阵中，并在相乘后从结果维度中消失。
- 普通维度：仅存在于其中一个矩阵中，并保留在相乘结果中。
- 举例：<font color="blue">B</font><font color="red">D</font> $\times$ <font color="blue">B</font><font color="red">D</font> $\to$ B
- **批量维度（蓝色标记）**：在两个操作数矩阵中都保留，并在结果中依然存在的维度。

![transformer-diagram](https://jax-ml.github.io/scaling-book/assets/img/transformer-diagram.png)

经典设计约定：
- $F = 4D$（MLP 块中的隐藏层宽度一般设为模型隐藏层维度的 4 倍）。
- $D = N \cdot H$（模型隐藏维度等于注意力头数 $N$ 乘以单头维度 $H$）。
- $N = K \cdot G$（在 GQA 分组查询注意力中，常规查询头数 $N$ 被划分为 $K$ 个分组，每一分组的 $G$ 个查询头共享同一组 KV 头）。
- $S = T$（在训练阶段，利用长度为 $S$ 的输入前缀预测长度为 $T$ 的输出预测）。

### 1.3 算术强度回顾 (Review of Arithmetic Intensity)

考虑一个典型的矩阵乘积运算：输入向量矩阵 $X \ (B \times D)$ 乘以权重矩阵 $W \ (D \times F)$。
- 其中 $B$ 为批量大小，$D$ 为隐藏层维度，$F$ 为 MLP 块中的上投影宽度。

让我们对该矩阵乘积 ($X \cdot W$) 进行 FLOPs 算力开销与 HBM 显存 IO 带宽读取字节数的精细盘点：
1. 从显存中读取输入矩阵 $X \ (B \times D)$：占用带宽 $2 \cdot B \cdot D$ 字节（以 bf16 混合精度训练为例，每个浮点数占 2 字节）。
2. 从显存中读取权重矩阵 $W \ (D \times F)$：占用带宽 $2 \cdot D \cdot F$ 字节。
3. 计算矩阵相乘 $Y = X \cdot W$：需要执行 $2 \cdot B \cdot D \cdot F$ 次浮点数计算（FLOPs）。
4. 将最终结果 $Y \ (B \times F)$ 写回显存：占用带宽 $2 \cdot B \cdot F$ 字节。

In [ ]:
# 执行上述盘点的 SymPy 验证
flops = 0
bytes_transferred = 0

# 1. 从显存读取 X
bytes_transferred += 2*B*D
# 2. 从显存读取 W
bytes_transferred += 2*D*F
# 3. 进行矩阵相乘
flops += 2*B*D*F
# 4. 将输出 Y 写回显存
bytes_transferred += 2*B*F

assert flops == 2*B*D*F
assert bytes_transferred == 2*B*D + 2*D*F + 2*B*F

回想一下，**算术强度 (Arithmetic Intensity)** 是指计算单元在处理数据时，平均每次显存读写（1 字节）能够支持的计算量（FLOPs）：
$$\text{算术强度} = \frac{\text{FLOPs}}{\text{Bytes Transferred}}$$
我们期望算法的算术强度越高越好，这有助于让计算芯片时刻保持在算力受限（Compute-bound）的最优能效比区间。

In [ ]:
# 计算并化简算术强度公式
intensity = (flops / bytes_transferred).simplify()
print("原始算术强度公式:", intensity)

# 假设在推理生成阶段，批量大小 B 远远小于模型的维度 D 和 F (B << D, F)
# 我们可以将 D 和 F 表示为以 B 为变量的极限形式来简化算术强度公式：
intensity_simplified = intensity.subs(D, c*B).subs(F, c*B).limit(c, oo).simplify()
print("极限简化后的算术强度:", intensity_simplified)
assert intensity_simplified == B

# 让我们以单张 H100 显卡为例估算硬件自身的算术强度临界线：
# H100 的 bf16 算力峰值为 989 TFLOPS (989e12)
# H100 的显存带宽为 3.35 TB/s (3.35e12)
flops_per_second = 989e12
memory_bandwidth_val = 3.35e12
accelerator_intensity = flops_per_second / memory_bandwidth_val
print(f"H100 显卡的算术强度分界线: {accelerator_intensity:.2f} FLOPs/Byte")
assert round(accelerator_intensity) == 295

对于分布式计算优化，有以下极简判定准则：
- 如果算法的**算术强度 $>$ 硬件临界值**，则属于 **算力受限 (Compute-bound)** 状态。我们可以完全发挥芯片所有的算力峰值。
- 如果算法的**算术强度 $<$ 硬件临界值**，则属于 **内存带宽受限 (Memory-bound)** 状态。即使芯片的算力再高，计算内核也在闲置等待数据从显存中拉取，实际性能完全被内存带宽限死。

结合我们在上方拟合得出的结果（在小批量大小下，矩阵乘法的算术强度极限近似等于其批量大小 $B$）：
- 只有当并发批量大小 $B > 295$ 时，矩阵乘积才能进入算力受限状态。
- 在大模型生成阶段的极端情况下（如 $B=1$ 单卡自回归推理，即矩阵-向量乘法）：
  - 此时算术强度近似为 **1**。
  - 这远远低于 H100 显卡的临界值 295，意味着 GPU 处于极端的内存带宽受限状态（读取了巨大的权重矩阵参数，却只对其进行了寥寥数次累加计算）。大语言模型的自回归推理正是面临这一致命瓶颈。

### 1.4 大模型推理的算术强度深度剖析 (Arithmetic Intensity of Inference)

参阅：[Scaling book chapter on inference](https://jax-ml.github.io/scaling-book/inference/)

![naive-inference](https://jax-ml.github.io/scaling-book/assets/img/naive-inference-1400.webp)

#### 1. 朴素推理流程
* 在自回归过程中，为了生成下一个新的 Token，如果盲目将历史中产生的所有 Token 重新传入 Transformer，会带来极其庞大的重复计算。
* 计算开销：生成 $T$ 个新 Token 稳健需要 $O(T^3)$ 的算力（单步前向时间为 $O(T^2)$ 级）。

#### 2. 引入 KV 缓存 (KV Cache) 的推理流程
* **关键认知**: 在前文自回归迭代中已经算好的 Query、Key 和 Value 特征在后续步骤中是不需要重复计算的。
* **对策**: 我们仅需在 HBM 显存中开辟一块持久空间来保存历史产生的 Key 和 Value 向量，即 **KV 缓存 (KV Cache)**。每次迭代我们仅需要对当前的单个新输入 Token 进行前向映射，并在自回归注意力计算中将其与显存中的 KV 缓存拼接即可。
  > [!NOTE]
  > **KV Cache 存储本质与计算逻辑澄清**：
  > - **到底存了什么**：KV Cache 存储的是历史所有 $S$ 个 Token 的 **Key 向量 ($K$) 和 Value 向量 ($V$)**（显存占用为 $2 \times S \times D$ 级）。它**不存储**过去的 Query 向量（算完即丢弃），也**不存储**中间的 QK 点积评分，更**不存在**“更新后的 V 值”（每个 Token 产生后的 $K$ 和 $V$ 是静态保留的）。
  > - **自回归时谁和谁点积**：当前新 Token 只产生自己的单个 Query 向量 $q_{new} \ (1 \times D)$，去和缓存中全部 $S$ 个 Key 向量做点积，得到 $1 \times S$ 的注意力评分，再对全部 $S$ 个 Value 向量做加权求和，将序列维度 $S$ 汇聚消解，重新输出为当前单个 Token 的表征向量 $(1 \times D)$。

![cached-inference](https://jax-ml.github.io/scaling-book/assets/img/cached-inference-1400.webp)

推理运行的两个差异巨大的物理阶段：
1. **Prefill (填充/预填充阶段)**: 瞬间加载并计算用户输入的 Prompt 文本，这一步可以全序列并行，类似于训练阶段。
2. **Generation (生成阶段)**: 串行自回归生成新 Token。每一步的自回归长度为 1（即 $T=1$）。

下面我们来定量盘点自回归推理中，**MLP 层**与**自注意力（Attention）层**的 FLOPs 与显存 IO 通信开销：
- 约定：$S$ 为已生成的历史序列长度，$T$ 为当前要处理/预测的 Token 数量。
- Prefill 阶段：$T = S$ (序列长度为 $S$)。
- Generation 阶段：$T = 1$。

#### MLP 块在推理生成阶段的步步盘点：
1. 从显存读取当前的输入激活值 $X \ (B \times T \times D)$，带宽开销为：$2 \cdot B \cdot T \cdot D$ 字节。
   > [!NOTE]
   > **输入激活值 $X$ 的物理来源与维度说明**：
   > - **物理来源**：这里的 $X$ 指的是当前 Transformer 块中，自注意力层完成计算后（对 Value 向量加权汇聚，经残差连接与 LayerNorm/RMSNorm 归一化），正式输入给 MLP 层的**当前 Token 的特征向量**。
   > - **维度为何是 $B \times T \times D$**：MLP 是逐位置独立运算（Position-wise），对历史上下文毫无感知。在自回归 Generation 阶段，当前步处理的 Token 数 $T = 1$，因此张量形状就是 $B \times 1 \times D$（完全不含历史上下文维度 $S$，因为 $S$ 已在注意力加权求和中被聚合消解）。课件保留字母 $T$ 是为了用统一步骤同时覆盖 Prefill 阶段（$T = S$）与 Generation 阶段（$T = 1$）。
   > - **常系数 `2` 的来源**：BF16 / FP16 半精度下每个浮点数元素占用 2 字节（2 Bytes）。相关基础概念请复习 **Lecture 02（资源核算）的“数值精度全景对比”与“算子访存字节数定量推导”章节**。
2. 从显存中读取 MLP 层的三组权重矩阵 $W_{up} \ (D \times F)$、$W_{gate} \ (D \times F)$ 和 $W_{down} \ (F \times D)$，带宽开销为：$3 \cdot 2 \cdot D \cdot F$ 字节。
   > [!NOTE]
   > **常系数 `6`（$3 \times 2$）的来源说明**：
   > - **乘数 `3`**：现代大模型（如 LLaMA、Mistral 等）普遍采用 SwiGLU 门控前馈网络，单层 MLP 包含 $W_{up}, W_{gate}, W_{down}$ 共 3 组形状为 $D \times F$ 的权重矩阵。
   > - **乘数 `2`**：每个权重参数在 BF16 半精度下占用 2 字节。
   > - 对应架构拓扑与参数推导细节请复习 **Lecture 03（模型架构与超参数）的“现代门控激活函数 SwiGLU 演进”与“传统 FFN vs 现代门控 FFN 拓扑对比”章节**。
3. 计算列分片权重矩阵乘法 $U = X \cdot W_{up}$：浮点计算量为 $2 \cdot B \cdot T \cdot D \cdot F$ FLOPs。
4. 将激活状态 $U$ 写入显存以备激活函数计算：带宽开销为 $2 \cdot B \cdot T \cdot F$ 字节。
5. 计算门控分支的矩阵相乘 $G = X \cdot W_{gate}$：浮点计算量为 $2 \cdot B \cdot T \cdot D \cdot F$ FLOPs。
6. 将激活状态 $G$ 写入显存：带宽开销为 $2 \cdot B \cdot T \cdot F$ 字节。
7. 计算门控激活值并与输出权重相乘 $Y = (\text{GeLU}(G) \odot U) \cdot W_{down}$：浮点计算量为 $2 \cdot B \cdot T \cdot D \cdot F$ FLOPs。
   > [!NOTE]
   > **矩阵维度与低阶项忽略说明**：
   > - **矩阵乘法维度**：融合中间张量 $(\text{GeLU}(G) \odot U)$ 的形状为 $(B \cdot T \times F)$，降维矩阵 $W_{down}$ 的形状为 $(F \times D)$。两者进行标准矩阵相乘输出 $(B \cdot T \times D)$，对应的乘加浮点计算量严格为 $2 \cdot (B \cdot T) \cdot F \cdot D = 2 \cdot B \cdot T \cdot D \cdot F$ FLOPs。
   > - **低阶项忽略原因**：乘法前的 $\text{GeLU}(G)$ 激活函数与 $\odot U$ 阿达马逐元素乘积，其计算量分别仅为 $\mathcal{O}(B \cdot T \cdot F)$。由于大模型的隐藏维度 $D$ 极大（如 LLaMA-2-13B 中 $D = 5120$），矩阵乘法项因包含因子 $2D$ 而比这两个逐元素操作大了 $2D \approx 10^4$ 倍（占该步总算力的 99.99% 以上）。按大模型 FLOPs 核算的通用学术与工程惯例，千分之一量级以下的逐元素低阶项直接忽略不计，仅统计占绝对支配地位的矩阵乘法 GEMM。
8. 将最终输出状态 $Y$ 写回显存以供下一层使用：带宽开销为 $2 \cdot B \cdot T \cdot D$ 字节。

In [ ]:
# 执行 MLP 的浮点开销与通信开销累加
flops = 0
bytes_transferred = 0

bytes_transferred += 2*B*T*D
bytes_transferred += 3 * 2*D*F
flops += 2*B*T*D*F
bytes_transferred += 2*B*T*F
flops += 2*B*T*D*F
bytes_transferred += 2*B*T*F
flops += 2*B*T*D*F
bytes_transferred += 2*B*T*D

assert flops == 6*B*T*D*F
assert bytes_transferred == 4*B*T*D + 4*B*T*F + 6*D*F

# 求解其算术强度并在 B*T << D, F 时进行极限化简
intensity = (flops / bytes_transferred).simplify()
intensity_simplified = intensity.subs(D, c*B*T).subs(F, c*B*T).limit(c, oo).simplify()
print("MLP 层在推理时的简化算术强度:", intensity_simplified)
assert intensity_simplified == B*T

对此，我们能清晰划分 MLP 块在不同阶段的表现：
1. **Prefill 阶段**：因为 $T$ 等同于 Prompt 序列长度，所以 $B \cdot T$ 非常大。MLP 层在前向填充时容易进入高性能的**算力受限**状态。
2. **Generation 阶段**：因为自回归是单步串行的，所以 $T = 1$。此时 MLP 的算术强度直接退化为并发序列数 $B$：
   - 如果线上的并发请求数 $B$ 非常小（例如低于 295），则即使是简单的 MLP 投影计算，也会掉入严重的显存带宽受限瓶颈中。

#### 自注意力机制 (Self-Attention) 在推理生成阶段的步步盘点：
我们假设采用高效融合的计算内核（如 FlashAttention）来消除 QK 乘积的中间大显存写回开销：
1. 从显存中读取当前的 Query 状态 $Q \ (B \times T \times D)$，以及历史保存的所有 $K$ 缓存 $K \ (B \times S \times D)$ 和 $V$ 缓存 $V \ (B \times S \times D)$：带宽开销为 $2 B T D + 4 B S D$ 字节。
   > [!NOTE]
   > **访存带宽与常系数说明**：
   > - **存储格式与精度一致**：Query 向量（$Q$）与 KV Cache（$K$ 和 $V$）均以 **BF16 格式**存储，每个元素严格占用 **2 字节**，两者的存储精度完全相同，并没有任何一方精度更高。
   > - **常系数 `4` 的物理来源**：KV Cache 同时包含 Key 和 Value 两个独立的张量，读取 $K$ 缓存需要 $2 \cdot B \cdot S \cdot D$ 字节，读取 $V$ 缓存同样需要 $2 \cdot B \cdot S \cdot D$ 字节，常系数 `4` 正是来自于 $K$ 矩阵的 2 字节与 $V$ 矩阵的 2 字节相加（即 $4 = 2_{\text{读}K} + 2_{\text{读}V}$）。
2. 计算注意力权重评分矩阵 $A = Q \cdot K^T$：浮点计算量为 $2 \cdot B \cdot S \cdot T \cdot D$ FLOPs。
3. 计算加权汇聚输出 $Y = \text{softmax}(A) \cdot V$：浮点计算量为 $2 \cdot B \cdot S \cdot T \cdot D$ FLOPs。
4. 将最终结果 $Y \ (B \times T \times D)$ 写入显存以备后用：带宽开销为 $2 \cdot B \cdot T \cdot D$ 字节。

In [ ]:
# 执行注意力层浮点与通信开销的累加
flops = 0
bytes_transferred = 0

bytes_transferred += 2*B*T*D + 2*B*S*D + 2*B*S*D
flops += 2*B*S*T*D
flops += 2*B*S*T*D
bytes_transferred += 2*B*T*D

assert flops == 4*B*S*T*D
assert bytes_transferred == 4*B*S*D + 4*B*T*D

# 求解其算术强度
intensity = (flops / bytes_transferred).simplify()
print("自注意力机制层在推理时的算术强度:", intensity)
assert intensity == S*T / (S + T)

In [ ]:
# 1. 评估 Prefill 阶段 (T = S)
prefill_intensity = intensity.subs(T, S).simplify()
print("Prefill 阶段自注意力算术强度:", prefill_intensity)
assert prefill_intensity == S/2

# 2. 评估 Generation 阶段 (T = 1)
generate_intensity = intensity.subs(T, 1).simplify()
print("Generation 阶段自注意力算术强度:", generate_intensity)

### 算术强度的关键性对比结论：
* **MLP 层**: 在 Generation 阶段，其算术强度等于并发 Batch 大小 $B$。我们可以通过在服务层进行大并发的“拼单组 Batch”（使 $B$ 接近或大于 295），来将 MLP 层拖出显存受限瓶颈。
* **注意力层**: 在 Generation 阶段，不管我们使用多大的并发批次 $B$，其算术强度公式为：
  $$\text{算术强度} = \frac{S}{S+1} < 1$$
  这意味着**注意力层无论如何也无法通过增大并发批次来改善其显存带宽瓶颈！**
  - **本质物理原因**: 在 MLP 计算中，不管 $B$ 怎么大，权重矩阵是所有并发序列公共的，拉取一次参数即可给所有序列做计算，因而实现带宽成本分摊。而在自注意力层中，每个并发请求都拥有完全不同的历史 KV 缓存，每增加一个 $B$，就需要额外从显存拉取对应大小的 $K$ 和 $V$ 状态，因而完全无法享受参数共享的分摊红利。

### 1.5 推理吞吐量与单步延迟估算 (Throughput and Latency)

至此，我们已经严格推导并证明了大模型自回归推理属于严重的**显存带宽受限（Memory-bound）**任务。
现在，我们可以在 **perfect latency-throughput tradeoff** 的理论上限假设下，定量计算单个请求的理论最高延迟和吞吐量：
* **延迟限制 (Latency)**: 单步生成时间主要受限于从 HBM 读取全部模型参数和当前已累积的所有 KV 缓存的总耗时（$\text{Latency} = \text{Memory} / \text{Memory Bandwidth}$）。
* **吞吐表现 (Throughput)**: 随着并发批次 $B$ 的增加，虽然单次迭代的 HBM 显存读写负荷加重导致单步延迟变高，但由于固定的模型权重被更多的序列所平摊，系统每秒输出的总 Token 数量会显著增加（$\text{Throughput} = B / \text{Latency}$）。

下面我们先定义 Transformer 的性能指标数据结构与计算函数，并配置标准的 Llama 2 13B 模型参数：

In [ ]:
@dataclass(frozen=True)
class TransformerPerformanceStats:
    """
    Transformer 的性能指标：
    - num_params：参数数量（以字节为单位）
    - memory：总显存占用（参数 + KV 缓存），以字节为单位
    - latency：生成一个 token 的时间（秒/token）
    - throughput：每秒生成的 token 数
    """
    num_params: int
    memory: int
    latency: float
    throughput: float

    def substitute(self, key, value):
        """在所有指标中将 `key` 替换为 `value`。"""
        return TransformerPerformanceStats(
            self.num_params.subs(key, value).simplify(),
            self.memory.subs(key, value).simplify(),
            self.latency.subs(key, value).simplify(),
            self.throughput.subs(key, value).simplify(),
        )


def compute_transformer_performance_stats(config) -> TransformerPerformanceStats:
    """根据给定的 `config` 计算 Transformer 的各项性能指标。"""
    # Transformer 中的权重参数数量
    num_params = 2*V*D + D*F*3*L + (2*D*N*H + 2*D*K*H)*L

    # 权重参数占用的内存大小 (bf16 每个参数占 2 字节)
    parameter_size = 2*num_params
    
    # 每个序列的 KV 缓存大小（S 个 token，K 个键/值头，H 维度，L 层，K+V 各占一份，每个参数 bf16 占 2 字节）
    kv_cache_size_per_seq = S * (K*H) * L * 2 * 2

    # 总内存占用
    memory = B * kv_cache_size_per_seq + parameter_size

    # 延迟由显存 IO 决定（每一步都需要从 HBM 读取所有权重参数 and KV 缓存）
    latency = memory / memory_bandwidth

    # 吞吐量是延迟的倒数，乘上并行序列数 B
    throughput = B / latency

    # 替换配置中的具体参数值
    num_params = num_params.subs(config).simplify()
    memory = memory.subs(config).simplify()
    latency = latency.subs(config).simplify()
    throughput = throughput.subs(config).simplify()

    return TransformerPerformanceStats(num_params, memory, latency, throughput)


def llama2_13b_config(args={}):
    return {
        S: 1024,   # 序列长度
        D: 5120,   # 模型维度
        F: 13824,  # 前馈维度
        N: 40,     # 查询头数量
        K: 40,     # 键/值头数量
        H: 128,    # 每个注意力头的维度
        L: 40,     # 层数
        V: 32000,  # 词表大小
        memory_bandwidth: 3.35e12,  # 显存带宽（H100 为 3.35 TB/s）
        **args
    }

在单张 H100 显卡（显存带宽 3.35 TB/s，显存容量 80GB）的理想配置下，我们计算 Llama 2 13B 在不同并发 Batch size 下的理论性能上限：

In [ ]:
# 加载 Llama 2 13B 配置
config = llama2_13b_config()
stats = compute_transformer_performance_stats(config)

# 1. 评估 Batch size = 1
b1 = stats.substitute(B, 1)
print("=== 并发数 B=1 ===")
print(f"参数总量: {b1.num_params / 1e9:.2f} B")
print(f"显存占用量: {b1.memory / 1e9:.2f} GB")
print(f"理论单步延迟上限: {b1.latency * 1000:.2f} 毫秒/token")
print(f"理论吞吐量上限: {b1.throughput:.2f} tokens/秒")

# 2. 评估 Batch size = 64
b64 = stats.substitute(B, 64)
print("\n=== 并发数 B=64 ===")
print(f"显存占用量: {b64.memory / 1e9:.2f} GB")
print(f"理论单步延迟上限: {b64.latency * 1000:.2f} 毫秒/token")
print(f"理论吞吐量上限: {b64.throughput:.2f} tokens/秒")

# 3. 评估 Batch size = 256
b256 = stats.substitute(B, 256)
print("\n=== 并发数 B=256 ===")
print(f"显存占用量: {b256.memory / 1e9:.2f} GB")
print(f"理论吞吐量上限: {b256.throughput:.2f} tokens/秒")

# 验证内存物理限制
h100_memory = 80e9
print(f"\n单卡 H100 显存大小: {h100_memory / 1e9:.2f} GB")
assert b256.memory > h100_memory  # 在 Batch=256 时，模型参数和庞大的 KV 缓存总量已经彻底撑爆了 H100 显存！

从实验拟合和物理开销推导中可以看出：
- 提高并发批次大小，**会恶化单步的响应延迟（Latency）**，因为需要读取更大体量的 KV 缓存。
- 提高并发批次大小，**能大幅提高系统的吞吐量（Throughput）**，因为能够极好地平摊模型参数读取这一固定巨额 HBM 开销。

因此，在企业级推理部署中，系统调优人员往往需要在单步交互延迟与系统总能效开销之间寻求折中平衡。如果想跨多机进一步做并行扩展：
- 简单并行：直接部署多路负载，模型多副本独立运行（延迟不变，系统总吞吐随副本数线性增加）。
- 困难并行：部署多卡张量并行（Tensor Parallel），将大模型参数和 KV 缓存分布切割在多张 GPU 上以容纳超大模型。

## 第二部分：走有损捷径技术 (Taking shortcuts - lossy)

**核心目标**：在可接受的精度损失范围内，大幅降低推理的计算与显存复杂度。

> [!NOTE]
> **第二部分总体知识脉络与核心逻辑图景**：
> 
> 在自回归 Generation 阶段，系统陷入严重的**内存带宽受限 (Memory-bound)**，单步延迟取决于 $\text{Latency} = \frac{\text{权重显存} + \text{KV 缓存显存}}{\text{显存带宽}}$。在硬件显存带宽固定的前提下，要想实现数倍的推理加速与显存释放，必须对分子挥刀。第二部分从三个完全不同的物理维度系统展开“有损捷径”：
> 
> ```
>                     【走有损捷径技术 (Lossy Shortcuts)】
>                                      │
>          ┌───────────────────────────┼───────────────────────────┐
>          ▼                           ▼                           ▼
> 【维度一：砍动态 KV 缓存】    【维度二：砍数值表示位宽】    【维度三：砍模型物理结构】
>    (2.1 架构级削减)             (2.2 精度级量化)             (2.3 结构级剪枝)
>          │                           │                           │
>   - GQA (多头分组共享)        - FP16/BF16 -> FP8/INT4     - 评估通道/头/层重要性
>   - MLA (低秩隐空间压缩)      - PTQ (离线校准: AWQ/GPTQ)   - 物理切除冗余结构
>   - CLA (跨层共享缓存)        - QAT (训练期注入噪声)       - 知识蒸馏 (Distill) 修复
>   - 滑动窗口 / 稀疏注意力
>          └───────────────────────────┬───────────────────────────┘
>                                      ▼
>                       【2.4 有损技术落地双配方】
>                       - 从头预训练配方 (如 GQA / MLA)
>                       - 事后改造修复配方 (如 AWQ / 剪枝蒸馏)
> ```
> 
> - **维度一（削减 KV 缓存）**：长文本高并发下，动态暴涨的 KV 缓存显存会迅速超越静态模型权重。通过 GQA（横向头共享）、MLA（低秩隐空间投影）、CLA（跨层共享）或滑动窗口，大幅降低每个 Token 需常驻显存的 KV 字节数。
> - **维度二（低精度量化）**：保持模型网络拓扑不变，将浮点数表示位宽从 16-bit（2B）压缩至 8-bit（1B）甚至 4-bit（0.5B），配合 AWQ / GPTQ 等敏感通道保护算法，在几乎无损精度的前提下使访存量成倍下降。
> - **维度三（结构剪枝与知识蒸馏）**：
  - **为什么剪（硬件友好性）**：GPU 的 Tensor Core 仅擅长稠密矩阵加速，随机把个别权重置零的“非结构化稀疏”无法在实际推理中带来速度收益。因此必须采用**结构化剪枝 (Structured Pruning)**——直接按物理单元“整块裁切”，如移除冗余注意力头 (Heads)、缩减 MLP 隐藏通道数 (Channels)、甚至砍掉整层网络 (Layers)，把 13B 模型物理缩减为 8B 尺寸，使每步自回归需从 HBM 搬运的静态参数量成比例骤降。
  - **怎么切（评估与物理裁剪）**：在少量校准集上，根据梯度、Loss 敏感度或激活值范数评估各注意力头/通道的重要性得分，切除贡献最低的行/列/层，得到一个维度更小、但依然保持规则矩阵形态的紧凑模型。
  - **为什么必须配合知识蒸馏 (Distillation)**：这种直接物理动刀的“粗暴外科手术”会严重破坏原模型的特征流动，造成剪枝瞬间模型性能断崖式坍塌。为了不花数百万美元从头重新预训练小模型，此时引入**“教师-学生 (Teacher-Student)”蒸馏**：以未剪枝的原完整大模型为 Teacher，指导裁剪后的紧凑小模型 (Student) 拟合其输出 Logits（软概率分布）和中间隐藏层特征。仅需极少量的校准数据与轻量微调，便能以极低算力成本迅速“治愈创口”，将小模型的精度恢复到逼近原大模型的水平。
> - **与第三部分“无损”的本质对比**：本部分技术在数学或结构上打破了原模型的等价性，通过轻微的精度牺牲换取极限的显存与速度红利；而第三部分的“投机采样”则在数学上完全保证输出分布与原模型 100% 严格一致（Lossless）。

### 2.1 削减 KV 缓存的技术手段 (Reduce KV Cache Size)

由于自回归生成严重受限于显存带宽与显存容量，首先考虑直接压缩 KV 缓存体积：

#### 1. 分组查询注意力 (Grouped-Query Attention, GQA)
参阅：[GQA 论文](https://arxiv.org/abs/2305.13245)

![GQA-diagram](https://jax-ml.github.io/scaling-book/assets/img/gmqa.png)

- **多头注意力 (MHA)**: 每个 Query 头都配有一对独立的 Key 头和 Value 头 ($K=N$)。KV 缓存开销极大。
- **多查询注意力 (MQA)**: 所有 Query 头强行共享同一对 Key 头 and Value 头 ($K=1$)。KV 缓存骤降，但对生成质量存在损伤。
- **分组查询注意力 (GQA)**: 对 Query 头进行分组，每一组头共享一对 KV 头。取得了生成精度与推理能效的绝佳权衡。

下面，我们对比 Llama 2 在常规 MHA 与采用 GQA 后，多并发下的显存与性能变化：

In [ ]:
# 1. 采用多头注意力 (MHA: K=40, B=64)
config_mha = llama2_13b_config({K: 40, B: 64})
stats_mha = compute_transformer_performance_stats(config_mha)
print(f"MHA (Batch=64) 显存占用: {stats_mha.memory / 1e9:.2f} GB")

# 2. 引入分组查询注意力 (GQA 1:5 头分组比例: K=8, B=64)
config_gqa = llama2_13b_config({K: 8, B: 64})
stats_gqa = compute_transformer_performance_stats(config_gqa)
print(f"GQA (Batch=64) 显存占用: {stats_gqa.memory / 1e9:.2f} GB")

# 3. 由于 GQA 极大释放了显存空间，我们可以从容将 Batch size 提升到 256
config_gqa_large = llama2_13b_config({K: 8, B: 256})
stats_gqa_large = compute_transformer_performance_stats(config_gqa_large)
print(f"GQA (Batch=256) 显存占用: {stats_gqa_large.memory / 1e9:.2f} GB")

# 验证内存物理限制
assert stats_gqa_large.memory < h100_memory  # 在 GQA 压缩下，Batch=256 成功塞入了单卡 80GB 的显存中！

#### 2. 多头潜在注意力 (Multi-Head Latent Attention, MLA)
参阅：[MLA 论文](https://arxiv.org/abs/2405.04434)

![MLA-schema](images/mla-schema.png)

- 传统的自注意力中，隐藏向量直接映射为 $N \cdot H$ 维的 Key 和 Value 并写入显存。
- MLA 的突破点：仅在显存中存储经低秩投影压缩后的潜在变量 $c \ (C \text{ 维度})$。在前向注意力计算时，临时将其投影上调至 $N \cdot H$ 维的 Key 和 Value。
- 例如：DeepSeek v2 将 $N \cdot H = 16384$ 的超大维度压缩至潜在特征空间 $C=512$，极大释放了长文本下的显存压力。
- 细节：为了解决低秩压缩与旋转位置编码（RoPE）不兼容的难题，MLA 为 Key 和 Value 额外开辟了 64 维特征通道，共计存储 $512 + 64 = 576$ 维，性能拟合对比依然极具性价比优势。
	> [!NOTE]
	> **深度解析：低秩压缩与 RoPE 的本质冲突及 DeepSeek 解耦破局机制**
	> 
	> **一、MLA 推理极限加速的命门：矩阵吸收 (Matrix Absorption)**
	> - 传统低秩投影在计算自注意力打分时，本需将缓存的 512 维隐向量 $c_i^{KV}$ 乘以升维矩阵 $W_{UK}$ 恢复出高维键向量 $k_i$：$\text{Score} = q_t k_i^T = q_t (c_i^{KV} W_{UK})^T$；
	> - **结合律吸收**：利用矩阵乘法结合律，可变换为：
	>   $$\text{Score} = (q_t W_{UK}^T) \cdot (c_i^{KV})^T$$
	>   升维矩阵 $W_{UK}^T$ 可**提前一次性“吸收”进当前步的单个 Query 向量中**（生成 $q_t' = q_t W_{UK}^T$）。遍历历史 $S$ 个 Token 时，直接拿 $q_t'$ 与缓存中的 512 维小向量做内积，**彻底省去了在每个历史 Token 上升维重构 $K$ 的巨大计算与带宽开销**。
	> 
	> **二、本质冲突：RoPE 为何会炸毁“矩阵吸收”？**
	> - 若在投影前引入 RoPE，历史第 $i$ 个 Token 的 Key 变为 $k_i^{\text{RoPE}} = R_i (c_i^{KV} W_{UK})$，点积公式退化为：
	>   $$\text{Score}_{t, i} = q_t^T R_t^T \cdot \mathbf{R_i W_{UK}^T} \cdot (c_i^{KV})^T$$
	> - **位置索引 $i$ 导致的算力爆炸**：旋转矩阵 $R_i$ 随历史每一个 Token 的位置 $i$ 实时变化，矩阵乘法不满足交换律，$W_{UK}^T$ 无法越过 $R_i$ 提前与 $q_t$ 结合。若强行先乘左侧项，由于带有指标 $i$，大矩阵乘法必须随历史序列长度 $S$ **重复计算 $S$ 次**，计算复杂度从 $\mathcal{O}(D \cdot d_c)$ 暴增为 $\mathcal{O}(S \cdot D \cdot d_c)$（4K 序列下计算量骤增数千倍），推理加速彻底失效。
	> 
	> **三、DeepSeek 破局方案：解耦位置编码 (Decoupled RoPE)**
	> - **物理维度切分**：将语义与位置彻底剥离，兵分两路：
	>   1. **纯语义通道（Content，512 维）**：低秩隐变量 $c_i^{KV}$ 绝不注入任何 RoPE，纯净无瑕，**100% 享受矩阵吸收红利**；
	>   2. **独立位置通道（Position，64 维）**：额外开辟轻量线性层投影出 64 维的独立键向量 $k_i^R$，**仅对这 64 维施加 RoPE 旋转**：$k_{i, \text{rope}}^R = R_i k_i^R$。
	> - **双路解耦打分公式**：
	>   $$\text{Score}_{t, i} = \underbrace{(q_{t, c} W_{UK}^T) \cdot (c_i^{KV})^T}_{\text{纯语义匹配分（矩阵吸收极速计算）}} + \underbrace{(R_t q_{t, r}) \cdot (R_i k_{i, r})^T}_{\text{相对位置偏置分（仅在 64 维小空间点积）}}$$
	> - **显存账本**：每个 Token 仅需存储 $512$（纯语义隐变量）$+ 64$（解耦位置向量）$= \mathbf{576}$ 维，相比传统 MHA（如 16384 维）**显存暴降 96.5%**！
	> 
	> **四、为什么只保留 64 维位置不会大幅损失性能？**
	> 1. **Value 向量原本就无位置**：在标准 Transformer 中，RoPE 本就从未施加在 $V$ 向量上，信息载荷表达完全未受削弱；
	> 2. **1D 标量距离容量饱和**：文本是一维序列，相对距离 $(t - i)$ 本质是标量差值。64 维拥有 $64 \div 2 = 32$ 个不同频率的正余弦旋转复平面，表征一维距离已高度饱和过剩；
	> 3. **消除位置对语义的污染**：传统全量 RoPE 会因大旋转角硬生生破坏远距离实体词的语义高相似度，解耦后“语义匹配度”与“空间距离衰减”各司其职，形成了更优越的归纳偏置。

在生成精度和测试对比上：
- 传统配置下，多头注意力 (MHA) 的收敛性能略优于分组查询注意力 (GQA)（但 GQA 便宜极多）。
- 而采用低秩重投影的 MLA，在相同显存开销下，其性能甚至小幅反超了昂贵的 MHA！

#### 3. 跨层注意力 (Cross-Layer Attention, CLA)
* 类似于 GQA 跨头共享 KV 状态的理念，CLA 在模型结构上**跨层（Layers）共享同一个 KV 缓存**，极具工程应用前景。
	> [!NOTE]
	> **深度解析：层语义不同，为什么能跨层复用 KV Cache？**
	> - **核心物理直觉（查同一个资料库，得出不同结论）**：
	>   - Transformer 各层的语义深度确实不同，但研究发现各层的 **Query 剧烈多变**（作为“主动寻找器”，每层带着本层特有的问题去检索历史），而相邻层的 **Key/Value 变化极其缓慢、余弦相似度极高**（作为历史序列的事实载荷，呈现严重表征冗余）。
	>   - 形象比喻：$K$（索引号）和 $V$（书本内容）构成了历史公共“图书馆”；不同深度的层就像不同学科的研究员，凭借各自独立的 $Q$ 算出专属的注意力权重分布，即使查阅同一个资料库也能提炼出完全不同的深浅上下文。
	> - **工程实现机制（以 2 层相邻共享为例）**：
	>   - **奇数层（主层）**：正常投影计算并存储当前 Token 的 $Q^{(1)}, K^{(1)}, V^{(1)}$，将 $K, V$ 写入 KV Cache；
	>   - **偶数层（副层）**：**只计算当前 Token 的 $Q^{(2)}$**，彻底跳过 $K^{(2)}, V^{(2)}$ 的投影计算与显存写入，直接读取上一层的 $K^{(1)}, V^{(1)}$ 计算注意力：
	>     $$\text{Attn}^{(2)} = \text{Softmax}\left(\frac{Q^{(2)} (K^{(1)})^T}{\sqrt{d_k}}\right) V^{(1)}$$
	>   - **显存与访存收益**：全模型的 KV Cache 显存占用与 HBM 搬运带宽**直接砍半（立省 50%）**！
	> - **为什么不会抹平层级语义深度？（三重独立机制保障）**：
	>   1. **$Q$ 严格独立**：每一层的 $Q$ 由本层隐状态独立生成，保证各层的注意力权重分布具有高度特异性；
	>   2. **输出投影 $W_O$ 严格独立**：注意力多头拼接（Concat）后的线性投影矩阵 $W_O \in \mathbb{R}^{D \times D}$（见 Lecture 03 与 Lecture 08）独立训练，负责多头信息的跨头融合（Inter-head Mixing）并将表征重新对齐至本层空间；
	>   3. **MLP 块全量独立运行**：占据模型 2/3 参数量与主要高阶非线性抽象能力的 MLP 层保持完全独立，充分消化并提炼本层语义。
	> - **与 GQA 的架构对仗**：GQA 是在**横向**（同一层不同注意力头间）共享 KV，CLA 则是沿**纵向**（深度相邻层之间）共享 KV。两者本质相同——**让 Query 保持独立以维持语义多样性，将冗余的 KV 极致压缩以拯救显存带宽**。

#### 4. 滑动窗口注意力 (Sliding Window Attention)
* 在大序列场景下，自回归仅与最近生成的局部序列产生密集关联。通过滑动窗口截断，将 KV 缓存控制在固定的常数窗口宽度，使其彻底与序列总长度 $S$ 解耦。

#### 5. DeepSeek v4 前沿稀疏注意力
* 针对 100 万超长序列，DeepSeek v4 提出了 CSA（压缩稀疏注意力）和 DSA（自适应稀疏注意力），自适应选择极少量特征写入显存，取得了卓越的推理加速效果。

### 2.2 混合精度与低精度量化 (Quantization)

在不改变模型网络拓扑的前提下，最简易的显存削减手段是**降低数据的数值表示精度**：
- **fp32 (4 字节)**: 具有完备的精度表现，一般用于模型训练期的权重积累与优化器计算。
- **bf16 (2 字节)**: 动态范围与 fp32 一致，目前是绝大多数大模型默认的预训练与推理精度。
- **fp8 (1 字节)**: 包括 e4m3 等多项数据格式，在最新的硬件（如英伟达 H100 架构）上可提供成倍提升的算力加速吞吐。
- **int8 / int4 (1 字节 / 0.5 字节)**: 常用于推理部署端的极低精度量化。

> [!NOTE]
> **深度解析：量化的数学本质与公式解密（为什么不能直接截断小数？）**
> 
> **1. 浮点截断 vs 整数定点**：
> - 若在浮点体系内转换（如 FP32 $\to$ BF16），直接截断尾数完全可行（因指数位自带动态缩放，可自适应调节量级）；
> - 但模型量化的终极目标是**定点整数（INT8/INT4）**，整数没有指数位和小数点。大模型权重集中在微小的连续区间（如 $[-0.1, +0.1]$），若直接四舍五入取整，所有权重将直接归零崩溃。
> 
> **2. 线性量化公式解密：$q = \text{round}(x / \text{Scale}) + \text{Zero\_Point}$**：
> - **`Scale`（比例尺/放大镜）**：将微小的连续浮点动态范围均匀拉伸并投射到离散整数区间。在工程上普遍采用**分组量化 (Per-Group Quantization)**，例如每 128 个权重共享一个独立的 Scale（仅需 2 字节 FP16 存储，显存开销微增约 3%，却能极度贴合局部极值）；
> - **`Zero_Point`（零点偏移）**：平移坐标轴以解决激活值非对称分布（如经 ReLU/GeLU 后全为正数）导致的整数刻度浪费问题，并确保浮点物理值 0.0 能无损精确对齐到某个整数格点。

下面，我们在 Python 下演示低精度线性量化与反量化的基础数学模型：

In [ ]:
x = 5.2342
scale = 0.1
zero_point = 4

# 1. 线性量化过程 (Float32 -> Int8)
x_quant = round(x / scale) + zero_point
print(f"量化后的 Int 整数值: {x_quant}")

# 2. 反量化还原过程 (Int8 -> Float32)
x_approx = (x_quant - zero_point) * scale
print(f"反量化还原出的近似浮点数: {x_approx:.4f}")
print(f"量化引入的绝对精度误差: {abs(x - x_approx):.4f}")

工业级量化调优技术类别：
1. **量化感知训练 (Quantization-Aware Training, QAT)**：在模型预训练或微调过程中，在前向计算中插入伪量化算子以注入量化噪声。训练出的模型对低精度极其鲁棒，但训练开销很高。
	> [!NOTE]
	> **深度解析：QAT 的核心矛盾与“双轨制”破局机制**
	> - **常见误区澄清**：QAT 并不是通过“在部分层加随机掩码”来让选中参数适应量化，而是为了攻克**量化取整操作不可导（梯度归零）**的本质矛盾。
	> - **底层矛盾（取整函数不可导）**：若直接将权重转为低精度整数进行训练，$q = \text{round}(w / \text{Scale})$ 是阶梯函数，其导数在绝大多数点处均为 0。若直接反向传播，$\frac{\partial \text{Loss}}{\partial w} = \frac{\partial \text{Loss}}{\partial q} \cdot \frac{\partial q}{\partial w} = 0$，梯度彻底归零，模型根本无法更新。
	> - **“双轨制”破解方案**：
	>   1. **浮点影子权重 (Master Weights)**：在显存中始终保留高精度的完整浮点权重（FP32/BF16）；
	>   2. **前向全量伪量化 (Fake Quantization)**：对所有目标层执行模拟量化再反量化：$\hat{w} = \text{Scale} \cdot \text{round}(w / \text{Scale})$。$\hat{w}$ 仍为浮点类型，但其数值已被强行拉扯至整数格点，真真切切注入了截断误差，使前向 Loss 能充分感知量化扰动；
	>   3. **反向直通估计 (Straight-Through Estimator, STE)**：反向求导时采用“直通假设”（$\frac{\partial \hat{w}}{\partial w} \approx 1$），梯度无损穿透伪量化层，直接更新底层的浮点影子权重，引导权重主动寻优至对离散截断高度鲁棒的参数空间；
	>   4. **离线部署**：训练收敛后，直接将浮点影子权重真正转为整数矩阵固化存储，上线脱离伪量化算子全速推理。

2. **后训练量化 (Post-Training Quantization, PTQ)**：在模型训练完成后，直接通过小样本校准集拟合数值的 Scale 和 Zero point 缩放系数。低成本，被广泛采用。
	> [!NOTE]
	> **深度解析：PTQ 两大流派（GPTQ vs AWQ）与通道本质**
	> 
	> **一、何为“通道 (Channel)”？**
	> - 在矩阵乘法 $Y = X \cdot W$ 中，输入激活值 $X \in \mathbb{R}^{B \times D_{\text{in}}}$ 的各特征分量，以及权重矩阵 $W \in \mathbb{R}^{D_{\text{in}} \times D_{\text{out}}}$ 的**行 (Rows)**，被称为**输入通道 (Input Channels)**；$W$ 的**列 (Columns)** 及输出 $Y$ 的分量被称为**输出通道 (Output Channels)**。
	> - 工业界研究发现：量化最敏感的**异常值尖峰 (Outliers)** 集中出现在极少数（约 0.1% ~ 1%）的**输入激活值通道**中。
	> 
	> **三、PTQ 两大救命流派的底层哲学分野**
	> - **GPTQ（参数代偿流——“刻度定死，全局分摊误差”）**：
	>   - **机制**：Scale 是一开始按极值直接定死的。当某参数被强行四舍五入为整数产生误差 $\Delta w_q = \hat{w}_q - w_q$ 时，GPTQ **绝不修改 Scale，而是直接修改其他尚未量化的浮点权重数值**！
	>   - **全局广播补偿**：该误差绝非仅由相邻列代偿，而是通过校准集计算出的 Hessian 协方差逆矩阵 $H^{-1} = (2 X^T X)^{-1}$，**一次性广播给当前块中所有剩余未量化的权重共同分摊**：
	>     $$w_{\text{remain}} \leftarrow w_{\text{remain}} - \frac{\Delta w_q}{[H^{-1}]_{qq}} \cdot (H^{-1})_{:, q}$$
	>     利用特征间的全连接协方差关系，在宏观输出层将局部截断误差完全抵消。
	> - **AWQ（刻度优化流——“保护精英，对角缩放恒等变换”）**：
	>   - **机制**：认为权重本身不应被扭曲，核心在于保护那 1% 携带关键语义的输入异常通道。
	>   - **数学实现（对角矩阵定向缩放）**：构造对角缩放矩阵 $S = \text{diag}(s_1, s_2, \dots, s_{D_{\text{in}}})$。对普通通道，对角线填 $1.0$（完全保持原样）；对关键敏感通道，对角线填大于 1 的放大系数 $s_i$。
	>   - **数学恒等式**：
	>     $$Y = X \cdot W = (X S^{-1}) \cdot (S W)$$
	>     - **权重侧（左乘对角阵 = 逐行缩放）**：$S \cdot W$ 将关键通道对应的权重**整行放大**，放大后四舍五入的相对截断误差大幅缩小；
	>     - **激活侧（右乘对角阵 = 逐列缩放）**：$X \cdot S^{-1}$ 将关键通道的激活值尖峰**整列缩小**，运行时直接融合进前序 LayerNorm，无任何额外计算开销；
	>     - 在数学上 100% 严格恒等，同时整张矩阵保持整齐划一的纯 INT4 密集形态，完美兼顾了 GPU 硬件加速与关键通道精度保护。

### 2.3 模型结构剪枝与知识蒸馏 (Model Pruning & Distillation)
参阅：[NVIDIA 剪枝论文](https://arxiv.org/abs/2407.14679)

![pruning-kd-loop](images/pruning-kd-loop.png)

* **核心思路**: 直接切除大模型中不重要的通道或层以缩减尺寸，随后利用原模型的知识蒸馏恢复性能。
* **三步算法流程**:
  1. 在小规模校准集（如 1024 个样本）上评估各个 {网络层, 注意力头, 隐藏维度} 的重要性得分；
  2. 移除得分最低的冗余结构，得到轻量化的子模型；
  3. 将原大模型的知识蒸馏（Distill）灌入裁剪后的子模型中。

### 2.4 有损方案设计配方小结 (Lossy Shortcuts Summary)
- **从头训练配方 (From-scratch recipe)**:
  1. 重新设计推理更快的全新架构（如 GQA/MLA） $\to$ 2. 在海量数据上直接从头预训练。
- **蒸馏修复配方 (Distillation recipe)**:
  1. 定义更快的模型架构 $\to$ 2. 使用原模型权重初始化新架构 $\to$ 3. 通过知识蒸馏修复精度损失。

## 第三部分：双重检验的无损捷径：投机采样 (Use shortcuts but double check - lossless)

参阅：[投机采样论文 1](https://arxiv.org/abs/2211.17192)，[论文 2](https://arxiv.org/abs/2302.01318)，[谷歌研究博客回顾](https://research.google/blog/looking-back-at-speculative-decoding/)

### 3.1 核心哲学：验证比生成快（非对称性）
回想推理的两大物理阶段：
- **Prefill (填充)**: 给定一串序列，并行编码所有 token（属于算力受限 Compute-bound，同时可得到各位置的条件概率）；
- **Generation (生成)**: 单步串行自回归生成单个 token（属于严重的内存受限 Memory-bound）。

**核心推论：让大模型“验证”一串候选词的速度，远快于大模型自己一步一步“生成”这串词！**

### 3.2 投机采样算法流程 (Speculative Sampling)
![speculative-sampling-algorithm](images/speculative-sampling-algorithm.png)

1. 使用开销低廉的**草稿模型 (Draft Model, $p$)** 快速自回归猜测生成若干个候选 Token（如 4 个）；
2. 将这组候选 Token 一次性送入强大的**目标大模型 (Target Model, $q$)** 并行前向计算，评估目标模型对候选词的接受概率；
3. 采用**修改版的拒绝采样 (Modified Rejection Sampling)** 机制判定是否接受候选词；一旦遇到拒绝点，立即由目标大模型重新采样纠正，并丢弃后续候选。

### 3.3 严格数学证明：为什么投机采样是 100% 精确的无损采样 (Exact Sample)？
假设词表中仅有两个词元 $\{A, B\}$：
- 目标模型真实概率分布为 $[q(A), q(B)]$；草稿模型预测分布为 $[p(A), p(B)]$；
- 假定草稿模型高估了 $A$（$p(A) > q(A)$），则由全概率公式必有 $p(B) < q(B)$；
- 残差概率修正项 $\max(q - p, 0)$ 为 $[0, q(B) - p(B)]$；
- **计算投机采样最终采出各个词元的实际边缘概率**：
  - 采出 $A$ 的概率：
    $$P[\text{sampling } A] = p(A) \cdot \frac{q(A)}{p(A)} + p(B) \cdot 1 \cdot 0 = q(A)$$
  - 采出 $B$ 的概率：
    $$P[\text{sampling } B] = p(B) \cdot 1 + p(A) \cdot \left(1 - \frac{q(A)}{p(A)}\right) \cdot 1 = p(B) + p(A) - q(A) = 1 - q(A) = q(B)$$

**数学结论**：无论草稿模型多么粗糙，最终生成的文本概率分布严格等价于**目标大模型直接串行采样（Exact Sample）**，数学上完全无损（Lossless）！

### 3.4 进阶扩展架构
- **Medusa**: 无需额外维护独立的草稿小模型，直接在目标大模型顶部附加多个轻量预测头并行预测后续 Token；
- **EAGLE**: 草稿模型直接复用目标大模型顶层的丰富隐藏特征，显著提高候选 Token 的被接受率。

## 第四部分：处理动态高并发工作负载 (Handling Dynamic Workloads)

在真实的线上云服务中，大模型推理面临请求长短不一、串行到达的 ragged array 挑战：

### 4.1 连续批次 (Continuous Batching)
* **旧式 DDP 推理**: 必须等到当前 Batch 中所有序列全部完成生成后，才能拉入下一批请求。这导致短文本请求要极度闲置等待长文本请求，造成极大的算力浪费。
* **新式迭代级调度**: 以单步迭代（Step-by-step）为基本调度单位。一旦某个请求触发了终止符（EOS），立即从当前批次中将其剔除，并将新到达的 Prompt 请求在下一次迭代中直接填充拉入并发批次。
	> [!NOTE]
	> **深度解析：为什么推理阶段可以“中途插队”？——Batch 语义的蜕变与连续批处理机制**
	> 
	> **一、Batch 语义的底层蜕变：从“数学优化单元”到“硬件访存拼车”**
	> - **训练阶段（强数学耦合）**：Batch 是损失聚合（Loss 求均值）与反向传播的最小单位。所有样本必须共同参与计算图前向传播并联合求导，以估计全量数据的期望梯度（$\mathbb{E}[\nabla \mathcal{L}]$）。样本间在算法层面被强行绑定，生命周期步调必须严格一致。
	> - **推理阶段（纯硬件体系结构语义）**：完全不存在 Loss 计算、反向求导与权重更新，各用户请求在数学逻辑上是 **100% 独立、零数学依赖（Zero Coupling）** 的。
	>   - **为什么还要按 Batch 组织？** 解决自回归解码的**内存带宽受限 (Memory-bound)** 瓶颈。单请求（$B=1$）每步生成一个 Token，GPU 都必须将全量模型权重（如 70B 模型占 140 GB 显存）从 HBM 读入 SRAM，算术强度仅为 $\approx 1\text{ FLOP/Byte}$，导致 GPU 计算单元 99.7% 的时间都在闲置等显存。
	>   - **推理 Batch 的物理本质**：让多个相互独立的请求**共享同一份只读权重的读取带宽（Weight Reuse）**，将低效的 GEMV 升格为密集的 GEMM，通过“并发拼车”大幅提升算术强度与吞吐能效。
	> 
	> **二、为什么可以随时退出并允许新序列“即时插队”？**
	> - **数学上的完全解耦**：既然样本之间不存在任何跨请求的数学计算依赖，传统推理强行让早结束的请求填充 `<PAD>` 占位符陪跑纯属继承自训练框架的机械惯性；
	> - **迭代级动态调度 (Iteration-level Scheduling)**：大模型推理的最小物理前向粒度是**单个 Token 的迭代步（Step）**。在每一步前向结束的纳秒级微隙：
	>   1. **即时退场**：输出 `<EOS>` 终止符的请求立即结算并释放并发槽位，彻底消除计算气泡（Bubbles）；
	>   2. **动态插队 (In-flight Insertion)**：新到达的请求无需等待长文本生成完毕，可立即被填入空闲槽位，无缝与正在解码的其他请求合并送入下一步前向计算。
	> - **核心收益**：在保证每个请求采样数学严格无损的前提下，系统吞吐量暴增 2~5 倍，且极大降低了新请求的首字等待延迟 (TTFT)。

### 4.2 分页注意力 (PagedAttention)
参阅：[vLLM 官方论文](https://arxiv.org/pdf/2309.06180.pdf)

* **历史痛点**: 显卡常规下必须为每个用户请求预分配一块连续的极大 KV 缓存显存空间（如按最大 Context 长度 4K 预配）。这导致了极度严重的显存碎片化瓶颈（包括内部碎片与外部空余碎片）。
* **虚拟内存分页机制**: 借鉴操作系统的内存管理设计。将每个序列的 KV 缓存划分到一系列**物理上非连续的内存块 (Blocks)** 中。系统维护一个逻辑块与物理块映射的 Page Table。在计算注意力时，虚拟映射读取，彻底消除了显存碎片。
* **前缀共享机制**: 极大简化了多并发下公共前缀（如系统 System Prompt 引导、多输出 Sample 探索）的显存共享，使多并发请求的 KV 缓存能够实现 Copy-on-Write 块级别的高效复用。
	> [!NOTE]
	> **深度解析：PagedAttention 软件层实现机理与“为何不固化进硬件”的权衡**
	> 
	> **一、软硬件归属：100% 软件层实现**
	> - **非 GPU 硬件行为**：Block 划分与 Page Table 纯属上层软件（如 vLLM 推理引擎）在应用层的自主构建，并非 GPU 硬件 MMU 的原生功能。
	> - **软件运行三步曲**：
	>   1. **预分配物理块池**：启动时一次性向 GPU 申请大块显存，并在软件层均匀切分为固定容量的物理块（如每个 Block 固定存放 16 个 Token 的 KV 向量）；
	>   2. **用户态页表映射**：调度器在 CPU 端动态维护每个请求的逻辑块到物理块的映射表（`Block_Table`），物理上非连续存储但在逻辑上连续；
	>   3. **自定义 CUDA 查表内核**：重写 Attention 计算算子（PagedAttention Kernel），GPU 线程在计算点积时现场根据逻辑 Token 序号计算块号与偏移，查表寻址读取物理显存，将显存浪费从 $60\% \sim 80\%$ 骤降至 $4\%$ 以下。
	> 
	> **二、为什么不直接固化到 GPU 硬件中？（体系结构权衡）**
	> 1. **查表开销被完全掩盖**：软件计算偏移仅需几个纳秒，相对于随后微秒级的访存和 GEMM 矩阵乘法占比不足 0.01%，已被 GPU 线程调度隐藏，固化到硬件几乎带来不了端到端加速；
	> 2. **语义鸿沟与高级逻辑**：PagedAttention 深度依赖前缀树缓存（Prefix Caching）、写时复制（CoW）和张量多维语义，通用的扁平字节硬件 MMU 无法承载如此复杂的策略；
	> 3. **算法敏捷性与防“暗硅”**：大模型架构演进以周为单位（如 MHA 迅速演化为 MLA、稀疏注意力 NSA），芯片 3~4 年的流片周期若把特定页表结构焊死在硅片上，极易迅速沦为无用的“暗硅”。保持“通用硬件算力 + 灵活软件定义”是最优工程选择。

## 第 10 讲核心总结 (Summary)

- 大模型推理的 Generation 阶段在物理上处于严重的内存带宽受限（Memory-bound）状态，这与预训练训练阶段截然不同。
- 推理优化的主干思路：**极力压缩显存中存留的 KV 缓存与权重参数大小，以提高算术强度分界线。**
- 主流技术阵营包括：
  1. **网络架构改良**: GQA, MLA 压缩 KV 缓存维度。
  2. **精度量化**: AWQ, GPTQ 等 fp8/int4 高效量化。
  3. **采样算法重构**: Speculative Sampling 投机采样，利用大模型验证并行的效率优势。
  4. **系统层面演进**: Continuous Batching 迭代级调度与 PagedAttention 虚拟分页内存管理。